# Dataset Check

Exploratory analysis of the extracted clips dataset.

**Goals:**
- Visualize class, distance, hand, and subject distributions
- Examine frame count distribution
- Decide and lock in the T window size for training
- Spot-check random clips visually
- Identify any data quality issues before training

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

PROJECT_ROOT = Path("../../")

METADATA_DIR = PROJECT_ROOT / "data" / "metadata" / "no_hardware"
CLIPS_DIR = PROJECT_ROOT / "data" / "clips" / "no_hardware"
ANNOTATIONS_CSV = METADATA_DIR / "annotations.csv"

df = pd.read_csv(ANNOTATIONS_CSV)
print(f"Loaded {len(df)} annotations from {ANNOTATIONS_CSV}")
df.head()

## Class Balance

How many clips per class? Ideal is balanced across all classes.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
class_counts = df["class"].value_counts().sort_index()
class_counts.plot(kind="bar", ax=ax, color="steelblue")
ax.set_title("Clips per class")
ax.set_xlabel("Class")
ax.set_ylabel("Number of clips")
ax.set_xticklabels(class_counts.index, rotation=0)
for i, v in enumerate(class_counts.values):
    ax.text(i, v + 1, str(v), ha="center")
plt.tight_layout()
plt.show()

print(f"\nClass balance ratio (max/min): {class_counts.max() / class_counts.min():.2f}")
print(f"Ideal: 1.00, Acceptable: <1.20, Poor: >1.50")

## Subject, Distance, and Hand Distribution

Check that no factor is over-represented.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Subject
subj_counts = df["subject_id"].value_counts().sort_index()
subj_counts.plot(kind="bar", ax=axes[0], color="seagreen")
axes[0].set_title("Clips per subject")
axes[0].set_ylabel("Number of clips")
axes[0].tick_params(axis="x", rotation=45)

# Distance
dist_counts = df["distance_m"].value_counts().sort_index()
dist_counts.plot(kind="bar", ax=axes[1], color="coral")
axes[1].set_title("Clips per distance (m)")
axes[1].set_ylabel("Number of clips")
axes[1].tick_params(axis="x", rotation=0)

# Hand
hand_counts = df["hand"].value_counts().sort_index()
hand_counts.plot(kind="bar", ax=axes[2], color="mediumpurple")
axes[2].set_title("Clips per hand")
axes[2].set_ylabel("Number of clips")
axes[2].tick_params(axis="x", rotation=0)

plt.tight_layout()
plt.show()

## Frame Count Distribution

This determines the T window size for training. We need all clips to be padded to a fixed length.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Histogram overall
axes[0].hist(df["frame_count"], bins=range(df["frame_count"].min(),
             df["frame_count"].max() + 2), color="steelblue", edgecolor="black")
axes[0].set_title("Frame count distribution (all clips)")
axes[0].set_xlabel("Frame count")
axes[0].set_ylabel("Number of clips")

# Per-class boxplot
classes = sorted(df["class"].unique())
data_by_class = [df[df["class"] == c]["frame_count"].values for c in classes]
axes[1].boxplot(data_by_class, labels=classes)
axes[1].set_title("Frame count by class")
axes[1].set_ylabel("Frame count")

plt.tight_layout()
plt.show()

print("\nFrame count statistics:")
print(df["frame_count"].describe())

print("\nPer class:")
print(df.groupby("class")["frame_count"].agg(["min", "max", "mean", "std"]))

## Choosing T (window size)

T must accommodate the longest clip with reasonable buffer.

**Decision rule:**
- T should be at least the max frame count
- Add ~10-15% buffer for variation in future subjects
- Round to a clean number for convenience

In [ ]:
max_frames = df["frame_count"].max()
p95_frames = int(df["frame_count"].quantile(0.95))
p99_frames = int(df["frame_count"].quantile(0.99))

print(f"Max frame count:        {max_frames}")
print(f"95th percentile:        {p95_frames}")
print(f"99th percentile:        {p99_frames}")

# Suggested T values
T_safe = int(np.ceil(max_frames * 1.15 / 4) * 4)  # 15% buffer, rounded to multiple of 4
T_p95 = int(np.ceil(p95_frames * 1.20 / 4) * 4)   # 20% buffer over p95

print(f"\nSuggested T (max + 15% buffer, multiple of 4): {T_safe}")
print(f"Suggested T (p95 + 20% buffer, multiple of 4): {T_p95}")
print(f"\nClips that would be truncated at T={T_p95}: {(df['frame_count'] > T_p95).sum()}")
print(f"Clips that would be truncated at T={T_safe}: {(df['frame_count'] > T_safe).sum()}")

## Visual Spot-Check

Load a few random clips and confirm the skeleton data looks reasonable.

In [ ]:
JOINT_NAMES = ["nose", "left_shoulder", "right_shoulder", "left_elbow",
               "right_elbow", "left_wrist", "right_wrist", "left_hip", "right_hip"]

def plot_clip_trajectories(clip_path, ax):
    """Plot wrist Y trajectories for a clip."""
    clip = np.load(clip_path)
    T = clip.shape[0]
    frames = np.arange(T)
    
    # Left wrist (idx 5), right wrist (idx 6)
    ax.plot(frames, clip[:, 5, 1], label="L wrist Y", color="red")
    ax.plot(frames, clip[:, 6, 1], label="R wrist Y", color="orange")
    ax.set_xlabel("Frame")
    ax.set_ylabel("Y position")
    ax.legend(fontsize=8)

# Pick one random clip per class
sample_clips = df.groupby("class").sample(n=1, random_state=42)

fig, axes = plt.subplots(1, 4, figsize=(20, 4))
for ax, (_, row) in zip(axes, sample_clips.iterrows()):
    clip_path = CLIPS_DIR / row["class"] / f"{row['clip_id']}.npy"
    plot_clip_trajectories(clip_path, ax)
    ax.set_title(f"{row['class']} | {row['hand']} | {row['frame_count']}f")

plt.tight_layout()
plt.show()

## Data Quality Checks

Confirm all clips load correctly and have expected shape.

In [ ]:
issues = []

for _, row in df.iterrows():
    clip_path = CLIPS_DIR / row["class"] / f"{row['clip_id']}.npy"
    
    if not clip_path.exists():
        issues.append((row["clip_id"], "FILE MISSING"))
        continue
    
    try:
        clip = np.load(clip_path)
    except Exception as e:
        issues.append((row["clip_id"], f"Load error: {e}"))
        continue
    
    # Shape check: (frames, 9, 3)
    if clip.ndim != 3 or clip.shape[1] != 9 or clip.shape[2] != 3:
        issues.append((row["clip_id"], f"Bad shape: {clip.shape}"))
        continue
    
    # Frame count match
    if clip.shape[0] != row["frame_count"]:
        issues.append((row["clip_id"],
            f"Frame mismatch: file={clip.shape[0]}, csv={row['frame_count']}"))
        continue
    
    # NaN check
    if np.isnan(clip).any():
        issues.append((row["clip_id"], "Contains NaN"))
        continue

print(f"Checked {len(df)} clips")
print(f"Issues found: {len(issues)}")
for clip_id, reason in issues[:10]:
    print(f"  {clip_id}: {reason}")

if not issues:
    print("\nAll clips passed quality checks ✓")

## Summary

Document the decisions made from this exploration.

In [ ]:
print("=" * 60)
print("DATASET SUMMARY")
print("=" * 60)
print(f"Total clips:        {len(df)}")
print(f"Classes:            {sorted(df['class'].unique())}")
print(f"Subjects:           {sorted(df['subject_id'].unique())}")
print(f"Distances (m):      {sorted(df['distance_m'].unique())}")
print(f"Hands:              {sorted(df['hand'].unique())}")
print(f"Frame count range:  {df['frame_count'].min()} - {df['frame_count'].max()}")
print(f"\nRecommended T window size: {T_safe}")
print("=" * 60)